In [ ]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
bank_marketing = fetch_ucirepo(id=222) 
  
# data (as pandas dataframes) 
X = bank_marketing.data.features 
y = bank_marketing.data.targets 
  
# metadata 
print(bank_marketing.metadata) 
  
# variable information 
print(bank_marketing.variables) 


# 1. Histogramy
Zrób histogram atrybutu duration dla całości danych. Następnie zrób analogiczne dwa histo-
gramy, osobno dla próbek pozytywnych i negatywnych (atrybut y równy yes albo no, odpo-
wiednio). Co ciekawego widać na tych wykresach?

In [ ]:
import plotly.express as px
import pandas as pd
bank_marketing_whole = pd.concat([X,y],axis=1)
fig_duration = px.histogram(bank_marketing_whole,x="duration",labels = {"duration" : "phone call duration [s]"},title = "Histogram for duration attribute")
fig_duration.update_layout(width=1500,height=1000, yaxis_title = "Number of calls")
fig_duration.update_traces(xbins=dict(
    start=0,
    end=5000,
    size=20 
))
fig_duration

In [ ]:
import polars as pl

fast_banks = pl.from_pandas(bank_marketing_whole)

yes = fast_banks.filter(pl.col("y")== "yes")
fig_yes = px.histogram(yes,x="duration",labels={"duration" : "phone call duration [s]"},title = "Duration of calls with customers who decided to open term deposit")
fig_yes.update_layout(width=1500,height=1000, yaxis_title = "Number of calls")
fig_yes.update_traces(xbins=dict(
    start=0,
    end=4000,
    size=20 
))
fig_yes.show()

In [ ]:


no = fast_banks.filter(pl.col("y") == "no")
fig_no = px.histogram(no,x="duration",labels={"duration" : "phone call duration [s]"},title = "Duration of calls with customers who decided not to open term deposit")
fig_no.update_layout(width=1500,height=1000, yaxis_title = "Number of calls")
fig_no.update_traces(xbins=dict(
    start=0,
    end=4000,
    size=20 
))
fig_no.show()

#### Na wykresach widać, że większość rozmów przyniosła negatywny skutek, ale czym dłuższa była rozmowa tym więcej klientów zdecydowało się na otwarcie lokaty

# 2.Histogram z suwakiem
Zrób histogram atrybutu balance dla osób powyżej 25 roku życia (atrybut age większy niż 25).
Zapoznaj się z pakietem ipywidgets, a następnie przekształć swój wykres w interaktywny z
suwakiem do wybierania granicy wieku i zobacz jak zmienia się histogram.

In [ ]:
from ipywidgets import interact



def plot(treshold = 25):
    balance_over_treshold = fast_banks.filter(pl.col("age") > treshold)
    fig = px.histogram(balance_over_treshold,x="balance",labels = {"balance" : "bank account balance [$]" }, title = f"Balance over {treshold}")
    fig.update_layout(width=1500,height=800, yaxis_title = "Number of clients")
    fig.show()
    
interact(plot,treshold=(0,100))





interactive(children=(IntSlider(value=25, description='treshold'), Output()), _dom_classes=('widget-interact',…

<function __main__.plot(treshold=25)>

# 3. Procent udanych rozmów
Dla ustalonego progu, t = 360, czasu trwania rozmowy (atrybut duration), wyznacz próbki
danych z atrybutem duration powyżej progu t. Policz ile procent z nich to próbki pozytywne.
Powtórz obliczenia dla różnych t i zrób wykres: na osi X próg t, na osi Y procent próbek
pozytywnych. Spróbuj wytłumaczyć zjawisko widoczne na wykresie.

In [ ]:
t=360
over_t = fast_banks.filter(pl.col("duration") > t)
count_over = over_t.height
count_over_yes = over_t.filter(pl.col("y")=="yes").height
percent_of_yes = (count_over_yes/count_over)*100





t_values = [0, 60, 120, 180, 240, 300, 360, 600, 900, 1200]


results = []

for t in t_values:
    over_t = fast_banks.filter(pl.col("duration") > t)
    count_over = over_t.height
    count_over_yes = over_t.filter(pl.col("y")=="yes").height
    percent_of_yes = (count_over_yes/count_over)*100
    results.append({"duration": t,"percent_of_yes": percent_of_yes})
results


df_results = pl.DataFrame(results)
df_results


fig = px.line(df_results,x="duration",y="percent_of_yes",labels={"duration": "treshold of call duration [s]","percent_of_yes" : "Success rate of calls [%]"}, title = "Success rate vs duration of call")
fig.update_yaxes(range=[0,100],dtick=10)
fig.show()

#### Widzimy, że czym dłuższa rozmowa, tym odsetek udanych rozmów rośnie

# 4
Powtórz poprzedni punkt dla atrybutu balance. Wyobraź sobie, że przeprowadzasz kampanię
marketingową lokat bankowych. Możesz kontaktować się z losowo wybranymi (z rozkładem
jednostajnym na całych danych) klientami banku, albo jedynie z klientami o saldzie (atrybut
balance) powyżej ustalonego progu (możesz sam ustalić ten próg). Która strategia jest lepsza?
Jaki próg wybrać?

In [ ]:
t=500
over_t = fast_banks.filter(pl.col("balance") > t)
count_over = over_t.height
count_over_yes = over_t.filter(pl.col("y")=="yes").height
percent_of_yes = (count_over_yes/count_over)*100





t_values = [-10000,-1000,0, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000]


results = []

for t in t_values:
    over_t = fast_banks.filter(pl.col("balance") > t)
    count_over = over_t.height
    count_over_yes = over_t.filter(pl.col("y")=="yes").height
    percent_of_yes = (count_over_yes/count_over)*100
    results.append({"balance": t,"percent_of_yes": percent_of_yes})
results


df_results = pl.DataFrame(results)
df_results

global_success_rate = fast_banks.select((pl.col("y")=="yes").mean()*100).item()
global_success_rate

fig = px.line(df_results,x="balance",y="percent_of_yes",labels={"balance": "treshold of balance [$]","percent_of_yes" : "Success rate of calls [%]"}, title = "Success rate vs balance")
fig.update_yaxes(range=[0,100],dtick=10)
fig.add_hline(
    y=global_success_rate, 
    line_dash="dash", 
    line_color="red", 
    annotation_text=f"Losowy wybór ({global_success_rate:.1f}%)", 
)
fig.show()


### Widzimy, że kontaktowanie się z klientami z dodatnimi saldami kont będzie bardziej efektywne i kontaktowanie się z losowo wybranymi klientami, ale powinniśmy wybrać punkt, gdzie pula klientów jest jeszcze wystarczająco duża 

# 5. Losowe zbioryy klientów
Podziel losowo zbiór danych na dwie części porównywalnej wielkości. Używając pierwszej części
danych, zrób wykres opisany w pkt. 4 i ustal próg (według własnego pomysłu). Przyjmij,
że klienci z saldem powyżej ustalonego progu powinni być zainteresowani lokatą bankową
(powinni być próbkami pozytywnymi). Oblicz ilu z nich, procentowo, nie było. Oblicz też
ilu zainteresowanych lokatą klientów miało saldo nie większe od ustalonego progu. Następnie
policz takie błędy modelu na drugiej części danych. Powtórz obliczenia dla różnych wartości
progów. Zrób wykres (według własnego pomysłu) prezentujący wyniki. Ostatecznie, to jaki
próg wybrać?

In [26]:
shuffled_banks = fast_banks.sample(fraction=1.0,shuffle=True,seed=5)

half_banks = shuffled_banks.height//2

part1 = shuffled_banks.head(half_banks)
part2 = shuffled_banks.tail(shuffled_banks.height - half_banks)


threshold = range(-1000,10000,500)
results = []

for t in threshold:
    part1_over_t = part1.filter(pl.col("balance")>t)
    if part1_over_t.height > 0:
        part1_no_over = (part1_over_t.filter(pl.col("y")=="no").height / part1_over_t.height)*100
    else:
        part1_no_over = 0.0

    part1_yes_whole = part1.filter(pl.col("y") == "yes")
    part1_yes_under = (part1_yes_whole.filter(pl.col("balance")<=t).height / part1_yes_whole.height)*100
    
    
    
    part2_over_t = part2.filter(pl.col("balance")>t)
    if part2_over_t.height > 0:
        part2_no_over = (part2_over_t.filter(pl.col("y")=="no").height / part2_over_t.height)*100
    else:
        part2_no_over = 0.0
    
    part2_yes_whole = part2.filter(pl.col("y") == "yes")
    part2_yes_under = (part2_yes_whole.filter(pl.col("balance")<=t).height / part2_yes_whole.height)*100
    
    
    results.append({
        "threshold" : t,
        "Part 1 - Clients over t who said no" : part1_no_over,
        "Part 1 - Clients under t who said yes": part1_yes_under,
        "Part 2 - Clients over t who said no": part2_no_over,
        "Part 2 - Clients under t who said yes": part2_yes_under
    })
    
df_results= pl.DataFrame(results)
df_melted = df_results.unpivot(index = "threshold", variable_name="Error type",value_name = "Error [%]")
    
fig = px.line(df_melted, x = "threshold",y="Error [%]",color = "Error type",title = "Error analysis by balance threshold")    
fig.show()
    

### Błędy mają bardzo wysoką wartość, ale stałą, niebieska / zielona linia to wskazuje na to że nieważne jaki próg wybierzemy jest to powyżej 80 procent. Jeśli wybierzemy próg ok. 3000, tutaj ryzyko tego, że klient odmówi, lub pominiemy klienta który by się zgodził wynoszą tyle samo, czyli 80 procent, ale jednocześnie rezygnujemy z 80 procent klientów co patrząc pod kątem biznesowym może nie być najlepszą decyzją.